# 05-03 RunnableLambda와 @chain 데코레이터: 함수를 체인으로 만들기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `RunnableLambda`로 파이썬 함수를 LCEL 파이프라인에서 사용할 수 있는 `Runnable` 객체로 변환하는 방법을 구현할 수 있어요
- 단일 인자 제약사항을 이해하고, 딕셔너리 래퍼 패턴으로 여러 인자를 처리하는 방법을 설명할 수 있어요
- `@chain` 데코레이터로 함수를 더 간결하게 `Runnable`로 변환하고, 두 방법의 차이를 비교할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- 파이썬 제너레이터(Generator)와 이터레이터(Iterator) 기초
- 파이썬 데코레이터(Decorator) 문법 (`@`)
- LCEL 파이프 연산자(`|`) 사용법

---

사용자 정의 함수를 LCEL 파이프라인에서 사용하는 방법은 두 가지가 있어요:

1. **`RunnableLambda`**: 함수를 `Runnable`로 명시적으로 래핑(Wrapping)
2. **`@chain` 데코레이터(Decorator)**: 함수에 데코레이터를 추가하여 자동으로 `Runnable`로 변환

두 방법 모두 기능적으로 동일하며, 상황에 따라 선택하여 사용할 수 있어요.

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


In [ ]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, chain
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")


## 1. RunnableLambda 기본 사용법

`RunnableLambda`는 사용자 정의 함수를 `Runnable`로 래핑하여 LCEL 파이프라인에서 사용할 수 있게 해줘요.

**중요한 제약사항:**

- 사용자 정의 함수는 **단일 인자만 받을 수 있어요**
- 여러 인자를 받는 함수를 사용하려면, 딕셔너리를 받아서 여러 인자로 풀어내는 래퍼(Wrapper) 함수를 작성해야 해요

> **주의:** `RunnableLambda(fn)`에 전달하는 함수가 2개 이상의 인자를 받으면 런타임 오류가 발생해요. 딕셔너리 하나를 받고 내부에서 값을 꺼내는 패턴으로 우회해요.

In [ ]:
# ============================================================
# TODO: RunnableLambda로 사용자 정의 함수를 체인에 연결하세요
# 힌트: RunnableLambda(fn) — fn은 반드시 단일 인자를 받아야 함
#       여러 인자가 필요하면 딕셔너리로 묶고 래퍼 함수를 사용
# 예상 결과: "3 + 9 equals 12."
# ============================================================

# ---------------------------------------------------
# RunnableLambda: 사용자 정의 함수를 Runnable로 래핑하기
# ---------------------------------------------------

# 1단계: 단일 인자를 받는 함수 정의
# RunnableLambda에 직접 전달 가능
def length_function(text: str) -> int:
    """텍스트의 길이를 반환하는 함수"""
    return len(text)


# 2단계: 두 인자를 받는 함수 (직접 RunnableLambda에 전달 불가)
def _multiple_length_function(text1: str, text2: str) -> int:
    """두 텍스트의 길이를 곱하는 함수 (2개 인자 — 직접 사용 불가)"""
    return len(text1) * len(text2)


# 3단계: 딕셔너리를 받는 래퍼 함수 (RunnableLambda에 전달 가능)
# 핵심: 딕셔너리 하나를 받고 내부에서 여러 값을 꺼내는 패턴
def multiple_length_function(_dict: dict) -> int:
    """딕셔너리에서 "text1"과 "text2"를 추출하여 길이를 곱하는 래퍼 함수"""
    return _multiple_length_function(_dict["text1"], _dict["text2"])


# 4단계: 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_template("what is {a} + {b}?")

# 5단계: 체인 구성
# - itemgetter: 딕셔너리에서 특정 키 추출 (Runnable로 자동 변환)
# - RunnableLambda: 사용자 정의 함수를 파이프라인에 연결


In [ ]:
# 체인 실행


## 2. RunnableConfig 활용

`RunnableLambda`는 선택적으로 `RunnableConfig`를 수용할 수 있어요.

이를 통해 콜백(Callback), 태그(Tag), 메타데이터(Metadata) 등 구성 정보를 중첩된 실행에 전달할 수 있어요.

> **실무 팁:** `RunnableConfig`를 사용하면 LangSmith(랭스미스) 추적에서 태그와 메타데이터가 함께 기록돼요. 복잡한 파이프라인에서 어떤 설정이 적용됐는지 추적할 때 유용해요.

In [ ]:
# ============================================================
# TODO: RunnableConfig를 받는 함수를 정의하고 RunnableLambda로 래핑하세요
# 힌트: def fn(text: str, config: RunnableConfig): ...  → 두 번째 인자로 config 수신
# 예상 결과: 잘못된 JSON을 LLM이 수정 시도 후 결과 출력
# ============================================================

from langchain_core.runnables import RunnableConfig
import json

# ---------------------------------------------------
# RunnableConfig: 태그, 메타데이터, 콜백을 중첩 체인에 전달
# ---------------------------------------------------


## 섹션 전환 — @chain 데코레이터로 더 간결하게

`RunnableLambda`로 함수를 래핑하는 방법을 배웠으니, 이제 같은 기능을 더 간결하게 표현하는 `@chain` 데코레이터를 살펴볼게요.

`@chain` 데코레이터를 사용하면 함수를 자동으로 `Runnable`로 변환할 수 있어요. 이는 `RunnableLambda`로 래핑하는 것과 기능적으로 동일하지만, 더 간결하고 읽기 쉬운 코드를 작성할 수 있어요.

In [ ]:
# ============================================================
# TODO: @chain 데코레이터를 사용하여 두 체인을 순차 실행하는 함수를 작성하세요
# 힌트: @chain 데코레이터를 함수 위에 붙이면 자동으로 Runnable로 변환됨
# 예상 결과: "양자역학" 입력 → 설명 생성 → 이모지 인스타그램 게시글 변환
# ============================================================

# ---------------------------------------------------
# @chain 데코레이터: 함수를 Runnable로 자동 변환
# ---------------------------------------------------

# 1단계: 프롬프트 템플릿 정의
prompt1 = ChatPromptTemplate.from_template("{topic}에 대해 짧게 한글로 설명해주세요.")
prompt2 = ChatPromptTemplate.from_template(
    "{sentence}를 이모지를 활용한 인스타그램 게시글로 만들어주세요."
)


# 2단계: @chain 데코레이터 적용
# @chain == @RunnableLambda 래핑과 동일하지만 더 간결하고 읽기 쉬움
# 데코레이터가 적용되면 invoke(), stream(), batch() 등 Runnable 메서드 자동 부여


In [ ]:
# @chain으로 만든 체인 실행


## 마무리 요약

### RunnableLambda vs @chain 데코레이터 비교

| 항목 | `RunnableLambda` | `@chain` 데코레이터 |
|------|-----------------|---------------------|
| 사용법 | `RunnableLambda(fn)` | `@chain` 데코레이터 추가 |
| 코드 간결성 | 보통 | 간결 |
| Runnable 메서드 | 사용 가능 | 사용 가능 |
| RunnableConfig 전달 | 가능 | 가능 |
| 적합한 상황 | 체인 중간 삽입 | 함수 자체를 체인으로 정의 |

### 핵심 요점

- `RunnableLambda`는 함수를 파이프라인 중간에 명시적으로 삽입할 때 사용해요
- 함수는 반드시 **단일 인자**를 받아야 해요. 여러 값이 필요하면 딕셔너리로 묶어서 전달해요
- `@chain` 데코레이터는 `RunnableLambda`와 동일한 기능이지만 더 읽기 쉬운 코드를 만들어줘요
- 두 방법 모두 `invoke()`, `stream()`, `batch()` 등 `Runnable` 메서드를 그대로 사용할 수 있어요

### 다음 노트북 예고

다음 노트북(`04-Routing.ipynb`)에서는 입력 데이터의 특성에 따라 서로 다른 처리 경로를 선택하는 라우팅(Routing) 패턴을 배워요. `RunnableLambda`가 조건부 분기에 어떻게 활용되는지 살펴볼게요.